# LAS CMS — Phase 3.3: RAG Preparation
---

In [1]:
import pandas as pd, numpy as np, json, re
from pathlib import Path
from datetime import datetime

CLEAN_DIR = Path("../outputs/cleaned")
OUTPUT_DIR = Path("../outputs/rag_ready")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

programs = pd.read_csv(CLEAN_DIR/"programs_cleaned.csv")
print(f"✓ programs: {len(programs):,}")
hearings = pd.DataFrame()
try: hearings = pd.read_csv(CLEAN_DIR/"hearings_cleaned.csv"); print(f"✓ hearings: {len(hearings):,}")
except: print("⚠ hearings not found")

✓ programs: 3,858
✓ hearings: 13,740


## Merge Hearings

In [2]:
if not hearings.empty and 'programsID' in hearings.columns:
    def agg(g):
        if 'date' in g.columns: g = g.sort_values('date')
        return " | ".join(f"Hearing {i} ({r.get('date','?')}): {str(r.get('hearingUpdate','')).strip()}"
               for i,(_,r) in enumerate(g.iterrows(),1) if str(r.get('hearingUpdate','')).strip() not in ('','nan'))
    
    ha = hearings.groupby('programsID').apply(agg).reset_index(); ha.columns=['programsID','hearing_history']
    hc = hearings.groupby('programsID').size().reset_index(); hc.columns=['programsID','total_hearings']
    
    programs = programs.merge(ha, left_on='id', right_on='programsID', how='left')
    programs = programs.merge(hc, left_on='id', right_on='programsID', how='left')
    programs['total_hearings'] = programs['total_hearings'].fillna(0).astype(int)
    programs['hearing_history'] = programs['hearing_history'].fillna('')
    print(f"✓ Merged into {(programs['total_hearings']>0).sum():,} cases")
else:
    programs['hearing_history']=''; programs['total_hearings']=0

✓ Merged into 1,741 cases


C:\Users\awais\AppData\Local\Temp\ipykernel_15472\4162757906.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ha = hearings.groupby('programsID').apply(agg).reset_index(); ha.columns=['programsID','hearing_history']


## Build Narratives & Export

In [5]:
def narrate(r):
    p = []
    # Case type — try clean version first, fall back to original
    for src in ['natureOfCase_clean', 'natureOfCase']:
        v = str(r.get(src, ''))
        if v not in ('', 'nan', 'UNKNOWN', 'None', 'UNSPECIFIED'):
            p.append(f"Case Type: {v}.")
            break
    
    for src, lbl in [('districtName', 'District'), ('levelOfCourt', 'Court')]:
        v = str(r.get(src, ''))
        if v not in ('', 'nan', 'UNKNOWN', 'None'): p.append(f"{lbl}: {v}.")
    
    g = str(r.get('gender', '')); a = r.get('age', '')
    if g not in ('', 'nan'):
        d = f"Client: {g}"
        if pd.notna(a) and str(a) != 'nan': d += f", age {a}"
        p.append(d + ".")
    
    for src, lbl in [('caseFileDate', 'Filed'), ('programName', 'Program')]:
        v = str(r.get(src, ''))
        if v not in ('', 'nan', 'None'): p.append(f"{lbl}: {v}.")
    
    f = str(r.get('caseFacts', ''))
    if f not in ('', 'nan', 'None') and len(f) > 5: p.append(f"Facts: {f}")
    
    hh = str(r.get('hearing_history', ''))
    if hh not in ('', 'nan') and len(hh) > 5: p.append(f"Hearings ({r.get('total_hearings', 0)}): {hh}")
    
    dec = str(r.get('caseDecision_clean', r.get('caseDecision', '')))
    if dec not in ('', 'nan', 'None'): p.append(f"Outcome: {dec}.")
    
    return " ".join(p)

programs['case_narrative'] = programs.apply(narrate, axis=1)
L = programs['case_narrative'].str.len()
print(f"✓ {len(programs):,} narratives | Avg: {L.mean():.0f} | Rich(500+): {(L>=500).sum():,}")

✓ 3,858 narratives | Avg: 2335 | Rich(500+): 2,127


In [6]:
# Chunk & Export
CHUNK_SIZE, OVERLAP = 800, 100
def chunk(text):
    if len(text)<=CHUNK_SIZE: return [text]
    sents = re.split(r'(?<=[.!?|])\s+', text)
    chunks, cur = [], ""
    for s in sents:
        if len(cur)+len(s)<=CHUNK_SIZE: cur+=(" "+s if cur else s)
        else:
            if cur: chunks.append(cur.strip())
            cur = (cur[-OVERLAP:]+" "+s) if OVERLAP and cur else s
    if cur: chunks.append(cur.strip())
    return chunks

def meta(r):
    m = {}
    if 'id' in r.index and pd.notna(r['id']): m['case_id'] = str(int(r['id']) if isinstance(r['id'],(int,float)) else r['id'])
    for src,key in [('natureOfCase_clean','case_type'),('caseDecision_clean','outcome'),
                     ('levelOfCourt','court_level'),('districtName','district'),('gender','gender')]:
        if src in r.index and pd.notna(r[src]):
            v=str(r[src]).strip()
            if v not in ('','nan'): m[key]=v
    if 'total_hearings' in r.index: m['total_hearings']=int(r['total_hearings'])
    m['length'] = len(str(r.get('case_narrative','')))
    return m

docs = []
for idx,row in programs.iterrows():
    n = str(row.get('case_narrative',''))
    if not n or n=='nan': continue
    m = meta(row)
    for ci,ch in enumerate(chunk(n)):
        docs.append({'id':f"{m.get('case_id',idx)}_{ci}",'text':ch,
                     'metadata':{**m,'chunk_index':ci,'source':'CMS_LAS'}})

print(f"✓ {len(docs):,} chunks")

with open(OUTPUT_DIR/"cms_cases_rag.jsonl",'w',encoding='utf-8') as f:
    for d in docs: f.write(json.dumps(d,ensure_ascii=False)+'\n')
with open(OUTPUT_DIR/"cms_cases_rag.json",'w',encoding='utf-8') as f:
    json.dump(docs,f,ensure_ascii=False,indent=2)
with open(OUTPUT_DIR/"cms_cases_readable.txt",'w',encoding='utf-8') as f:
    for _,row in programs.iterrows():
        n=str(row.get('case_narrative',''))
        if n and n!='nan':
            m=meta(row)
            f.write(f"{'═'*60}\nCase {m.get('case_id','?')} | {m.get('case_type','?')} | {m.get('outcome','?')}\n{'─'*60}\n{n}\n\n")
pd.DataFrame([meta(row) for _,row in programs.iterrows()]).to_csv(OUTPUT_DIR/"cms_metadata.csv",index=False)

print(f"""
{'='*60}
  ALL PHASES COMPLETE!
{'='*60}
  Phase 2:   ✓ EDA — {len(programs):,} cases
  Phase 3.1: ✓ PII removed
  Phase 3.2: ✓ 8 tasks done
  Phase 3.3: ✓ RAG — {len(docs):,} chunks

  📁 ../outputs/rag_ready/
     cms_cases_rag.jsonl  ← VECTOR STORE INPUT
""")

✓ 16,511 chunks

  ALL PHASES COMPLETE!
  Phase 2:   ✓ EDA — 3,858 cases
  Phase 3.1: ✓ PII removed
  Phase 3.2: ✓ 8 tasks done
  Phase 3.3: ✓ RAG — 16,511 chunks

  📁 ../outputs/rag_ready/
     cms_cases_rag.jsonl  ← VECTOR STORE INPUT

